# Unlinked-observation overpass plan (Q3 Aqua → NOAA-20)

Select the actual Q3 Aqua sample observation marked `no_candidate_link_under_1km_3h`, then search the saved 2025 NASA CMR Level-2 manifest for a NOAA-20 VJ114IMG granule whose time interval is within ±3 hours and whose CMR polygon contains the Aqua coordinate. Select the temporally nearest coverage candidate, with granule ID as deterministic tie-break. This plan retrieves only one NOAA-20 L2 granule so its actual native fire mask and exact geolocation lineage can be inspected. Coarse CMR polygon containment is only a metadata filter, not proof of pixel opportunity.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/reference').is_dir())
sys.path.insert(0, str(ROOT))
from src.firewatch.validation_sampling import _cmr_contains
sample_path = ROOT/'data/processed/source_pixel_validation_2025_stratified.csv'
manifest_path = ROOT/'data/reference/cmr_manifest_2025.csv'
plan_path = ROOT/'data/reference/unlinked_opportunity_l2_plan_2025.csv'
sample = pd.read_csv(sample_path)
aqua = sample[(sample.quarter=='Q3') & sample.satellite.eq('Aqua') & ~sample.candidate_status & sample.candidate_status_label.eq('no_candidate_link_under_1km_3h')].iloc[0]
manifest = pd.read_csv(manifest_path)
manifest['time_start_utc'] = pd.to_datetime(manifest.time_start_utc,utc=True)
manifest['time_end_utc'] = pd.to_datetime(manifest.time_end_utc,utc=True)
obs_time = pd.Timestamp(aqua.observation_time_utc)
window_start, window_end = obs_time-pd.Timedelta(hours=3), obs_time+pd.Timedelta(hours=3)
candidates = manifest[manifest.short_name.eq('VJ114IMG') & manifest.time_start_utc.le(window_end) & manifest.time_end_utc.ge(window_start)].copy()
candidates = candidates[candidates.spatial_polygons.map(lambda p:_cmr_contains(p,float(aqua.csv_longitude),float(aqua.csv_latitude)))].copy()
assert not candidates.empty, 'No NOAA-20 L2 granule passes time and coarse polygon filters'
candidates['center_time_utc'] = candidates.time_start_utc+(candidates.time_end_utc-candidates.time_start_utc)/2
candidates['center_delta_minutes'] = (candidates.center_time_utc-obs_time).abs().dt.total_seconds()/60
candidates = candidates.sort_values(['center_delta_minutes','producer_granule_id'],kind='stable')
selected = candidates.iloc[0]
display(pd.DataFrame([{'Aqua_observation_utc':aqua.observation_time_utc,'Aqua_latitude':aqua.csv_latitude,'Aqua_longitude':aqua.csv_longitude,'Aqua_candidate_status':aqua.candidate_status_label,'NOAA20_L2_granule':selected.producer_granule_id,'NOAA20_start_utc':selected.time_start_utc,'NOAA20_end_utc':selected.time_end_utc,'center_delta_minutes':selected.center_delta_minutes,'CMR_size_MiB':selected.granule_size_cmr_mb}]))

,Aqua_observation_utc,Aqua_latitude,Aqua_longitude,Aqua_candidate_status,NOAA20_L2_granule,NOAA20_start_utc,NOAA20_end_utc,center_delta_minutes,CMR_size_MiB
0,2025-09-18T14:06:00+00:00,4.0848,9.3146,no_candidate_link_under_1km_3h,VJ114IMG.A2025261.1330.002.2025261205555,2025-09-18 13:30:00+00:00,2025-09-18 13:36:00+00:00,33.0,2.705573


In [2]:
# Keep the file transfer to one NOAA-20 Level-2 granule. Preserve query lineage and source-point context.
plan = pd.DataFrame([{
    'data_url': selected.data_url,
    'granule_size_cmr_mb': float(selected.granule_size_cmr_mb),
    'producer_granule_id': selected.producer_granule_id,
    'short_name': selected.short_name,
    'collection_version': selected.collection_version,
    'cmr_entry_id': selected.cmr_entry_id,
    'source_observation_id': aqua.observation_id,
    'source_observation_time_utc': aqua.observation_time_utc,
    'source_observation_latitude': float(aqua.csv_latitude),
    'source_observation_longitude': float(aqua.csv_longitude),
    'source_candidate_status': aqua.candidate_status_label,
    'selection_method': 'nearest NOAA-20 Level-2 granule center within ±3 h and CMR polygon containment; deterministic granule-ID tie-break',
    'center_delta_minutes': float(selected.center_delta_minutes),
    'granule_start_utc': selected.time_start_utc.isoformat(),
    'granule_end_utc': selected.time_end_utc.isoformat(),
    'spatial_polygons': selected.spatial_polygons,
}])
plan.to_csv(plan_path,index=False)
meta={'parent_cmr_manifest':manifest_path.name,'parent_manifest_sha256':__import__('hashlib').sha256(manifest_path.read_bytes()).hexdigest(),'plan_sha256':__import__('hashlib').sha256(plan_path.read_bytes()).hexdigest(),'selected_rows':1,'granules_downloaded':0,'coarse_polygon_is_pixel_coverage':False}
plan_path.with_suffix('.metadata.json').write_text(json.dumps(meta,indent=2),encoding='utf-8')
display(plan[['short_name','producer_granule_id','granule_size_cmr_mb','source_candidate_status','center_delta_minutes']])
print('Plan:',plan_path.relative_to(ROOT)); print('Metadata-only selection; no network request or granule download performed.')

,short_name,producer_granule_id,granule_size_cmr_mb,source_candidate_status,center_delta_minutes
0,VJ114IMG,VJ114IMG.A2025261.1330.002.2025261205555,2.705573,no_candidate_link_under_1km_3h,33.0


Plan: data\reference\unlinked_opportunity_l2_plan_2025.csv
Metadata-only selection; no network request or granule download performed.
